Dataset preprocessing

In [ ]:
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from tools.preprocessor import NLProcessor
import pandas as pd
import numpy as np
import torch

df1 = pd.read_csv("data/intents_train.csv")
df2 = pd.read_csv("data/bike_shop_intents.csv")
df3 = pd.read_csv("data/bike_shop_intents2.csv")


df = pd.concat([df1, df2, df3], ignore_index=True)

print("data frame shape:", df.shape)

del df1, df2, df3

text_train, text_val, intent_train, intent_val = train_test_split(
    df['text'],
    df['intent'],
    test_size=0.1,
    stratify=df['intent'],
    random_state=42
)

processor = NLProcessor(use_stopwords=True)
le        = LabelEncoder()

X_train  = processor.transform(text_train).astype(np.float32)
y_train  = le.fit_transform(intent_train).astype(np.int64)

y_train  = np.delete(y_train, processor.unfound_words)
processor.unfound_words = []

X_train = torch.from_numpy(X_train)
y_train = torch.from_numpy(y_train)
classes_ = le.classes_

print("y classes :", le.classes_)
print("X train shape:", X_train.shape)
print("y train shape:", y_train.shape)

X_val = processor.transform(text_val).astype(np.float32)
y_val = le.transform(intent_val).astype(np.int64)
y_val = np.delete(y_val, processor.unfound_words)

X_val = torch.from_numpy(X_val)
y_val = torch.from_numpy(y_val)

print("X val shape:", X_val.shape)
print("y val shape:", y_val.shape)

train_loader = DataLoader(
    TensorDataset(X_train, y_train),
    batch_size=32
)

data frame shape: (333, 2)
y classes : ['Bike Types' 'Cost Estimation' 'Hours' 'Make Appointment' 'Return Policy'
 'Trade-in Options' 'Welcome Intent']
X train shape: torch.Size([296, 7, 300])
y train shape: torch.Size([296])
X val shape: torch.Size([34, 7, 300])
y val shape: torch.Size([34])


RNN Solution

In [7]:

from torch.utils.data import DataLoader, TensorDataset
from tools.rnn import RNN
import torch.nn as nn
from tools.earlystopping import EarlyStopping
from tools.train import train

model = RNN(
    input_size=X_train.shape[2],
    hidden_size=128,
    num_layers=2,
    num_classes=len(classes_),
    dropout_rate=0.2
)
patience = 10
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001, weight_decay=1e-4)
earlystopping = EarlyStopping(patience=patience)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode="min", factor=0.5, patience=patience
)

num_epochs = 100
for epoch in range(num_epochs):

    model.train()
    avg_train_loss, train_acc = train(model, criterion, optimizer, train_loader)

    print(f"epoch {epoch} / {num_epochs} | ", end='')
    print(f"train loss : {avg_train_loss} | train acc : {train_acc}")

    model.eval()
    val_loss, val_acc = EarlyStopping.compute_val_metrics(model, criterion, X_val, y_val)

    scheduler.step(val_loss)

    print(f"val loss : {val_loss} | val acc : {val_acc}")

    if earlystopping(val_loss, model):
        break
    

torch.save(model, "models/rnn.pkl")
print("RNN model saved as models/rnn.pkl")

epoch 0 / 100 | train loss : 1.9279451727867127 | train acc : 0.17567567567567569
val loss : 0.05449600780711455 | val acc : 0.29411764705882354
epoch 1 / 100 | train loss : 1.6360942721366882 | train acc : 0.3547297297297297
val loss : 0.04667177971671609 | val acc : 0.3235294117647059
epoch 2 / 100 | train loss : 1.3953397512435912 | train acc : 0.39864864864864863
val loss : 0.03918122544008143 | val acc : 0.4411764705882353
epoch 3 / 100 | train loss : 1.2134138107299806 | train acc : 0.5202702702702703
val loss : 0.037216803606818706 | val acc : 0.4411764705882353
epoch 4 / 100 | train loss : 1.0595598340034484 | train acc : 0.5777027027027027
val loss : 0.034350956187528724 | val acc : 0.5
epoch 5 / 100 | train loss : 0.9536275863647461 | train acc : 0.5878378378378378
val loss : 0.03111513572580674 | val acc : 0.6176470588235294
epoch 6 / 100 | train loss : 0.8091596484184265 | train acc : 0.6587837837837838
val loss : 0.029278963804244995 | val acc : 0.7058823529411765
epoch 7 

LSTM Solution

In [ ]:
from tools.lstm import LSTM

model = LSTM(
    input_size=X_train.shape[2],
    hidden_size=128,
    num_layers=2,
    num_classes=len(classes_),
    dropout_rate=0.5
)

optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
criterion = nn.CrossEntropyLoss()
earlystopping = EarlyStopping()

num_epochs = 29
for epoch in range(num_epochs):

    model.train()
    train(model, criterion, optimizer, train_loader)

    model.eval()
    val_loss, val_correct = EearlyStopping.compute_val_metrics(model, criterion, X_val, y_val)
    ## print val metrics here!
    if earlystopping(val_loss, model):
        break

torch.save(model, "models/lstm.pkl")
print("LSTM model saved as models/lstm.pkl")

CNN Solution

In [ ]:
from tools.cnn import TextCNN

X_train = X_train.permute(0, 2, 1)
X_val   = X_val.permute(0, 2, 1)

batch_size, embed_dim, seq_len = X_train.shape
dataloader = DataLoader(
    TensorDataset(X_train, y_train),
    batch_size=32
)
# ------------------------------------------------------------------
# Model, loss, optimiser, scheduler
# ------------------------------------------------------------------
model = TextCNN(
    embed_dim=embed_dim,
    out_channels=128,
    output_size=len(classes_),
    kernel_sizes=[3, 5, 7],
    dropout_rate=0.5,
)
 
criterion = nn.CrossEntropyLoss(label_smoothing=0.05)  # mild label smoothing
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-4)
# Reduce LR by ×0.5 whenever train loss stops improving for 3 epochs
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode="min", factor=0.5, patience=3
)
earlystopping = EarlyStopping()

epochs = 30
for epoch in range(num_epochs):

    model.train()
    train(model, criterion, optimizer, train_loader)

    model.eval()
    val_loss, val_correct = EearlyStopping.compute_val_metrics(model, criterion, X_val, y_val)
    ## print val metrics here!
    if earlystopping(val_loss, model):
        break

torch.save(model.state_dict(), "models/text_cnn.pkl")
print("CNN saved as models/text_cnn.pkl")

BERT Solution

In [ ]:
from transformers import BertTokenizer
from tools.bert import IntentClassifier
import torch
import torch.nn as nn
from sklearn.metrics import accuracy_score

tokenizer = BertTokenizer.from_pretrained(
    "bert-base-uncased"
)

le = LabelEncoder()
y_train  = torch.from_numpy(le.fit_transform(intent_train).astype(np.int64))

encoding = tokenizer(
    text_train.values.tolist(),
    padding="max_length",
    truncation=True,
    max_length=32,
    return_tensors="pt"
)


model = IntentClassifier(num_classes=len(le.classes_))
optimizer = torch.optim.Adam(model.parameters(), lr=2e-5)
criterion = nn.CrossEntropyLoss()


X_loader = DataLoader(TensorDataset(encoding['input_ids'], encoding['attention_mask']),
    shuffle=False,
    batch_size=32
)

y_loader = DataLoader(
    y_train,
    shuffle=False,
    batch_size=32
)

model.train()
num_epochs = 10
for epoch in range(num_epochs):
    train_loss = 0
    train_correct = 0
    for (batch_inputs_ids, batch_attention_mask), batch_label in zip(X_loader, y_loader):
        optimizer.zero_grad()
        outputs = model(batch_inputs_ids, batch_attention_mask)

        loss = criterion(outputs, batch_label)
        loss.backward()

        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()

        train_loss += loss.item()
        train_correct += (outputs.argmax(dim=1) == batch_label).sum().item()


    avg_train_loss = train_loss / len(X_loader)
    train_acc      = train_correct / len(X_loader.dataset)
    print(f"Epoch [{epoch+1}/20] Loss: {train_loss/len(X_loader):.4f}")

    print(
        f"Epoch [{epoch+1}/{num_epochs}] "
        f"Train Loss: {avg_train_loss:.4f}  Acc: {train_acc:.3f}"
    )

test_encoding = tokenizer(
    text_test.values.tolist(),
    padding="max_length",
    truncation=True,
    max_length=32,
    return_tensors="pt"
)
y_test = le.fit_transform(intent_test).astype(np.int64)
preds = None
try:
    with torch.no_grad():
    
        outputs = model(test_encoding['input_ids'], test_encoding['attention_mask'])
        preds   = torch.argmax(outputs, dim=1)
    
    print("Text BERT + FNN accuracy score:",accuracy_score(
            y_true=y_test,
            y_pred=preds.numpy()
        )
    )
except Exception as e:
    print(str(e))
torch.save(model.state_dict(), "models/bert_fnn.pkl")
print("BERT + FNN saved as models/bert_fnn.pkl")




Inference

In [ ]:
try:
    with torch.no_grad():
    
        outputs = model(test_encoding['input_ids'], test_encoding['attention_mask'])
        preds   = torch.argmax(outputs, dim=1)
    
    print("Text BERT + FNN accuracy score:", accuracy_score(
            y_true=y_test,
            y_pred=preds.numpy()
        )
    )
except Exception as e:
    print(str(e))

# test_df = pd.read_csv("data/intents_test.csv")

# X_test = processor.transform(test_df['text']).astype(np.float32)

# model = torch.load("models/text_cnn.pkl")
# model.eval()

# X_test = X_test.permute(0, 2, 1)

# preds = None
# with torch.no_grad():
    
#     outputs = model(X_test)
#     preds   = torch.argmax(outputs, dim=1)
    
# df = pd.DataFrame({'intent':preds.numpy().to_list()})
# df.save("intents.csv")